In [3]:
import pandas as pd
from sodapy import Socrata

In [ ]:
ridership_data_sources = {
    "ridership_2025_2026": '5wq4-mkjj',
    "ridership_2020_2024": 'wujg-7c2s',
    "ridership_2017_2019": 't69i-h2me',
}
od_data_sources = {
    "od_2021": 'rapa-97zv',
    "od_2025": 'y2qv-fytt',
    "od_2026": '28vm-gjqr'
}

In [ ]:
APP_TOKEN = ""
def query_open_data(dataset_id, query):
    client = Socrata("data.ny.gov", APP_TOKEN, timeout=10000)

    try:
        results = client.get(dataset_id, query=query)

        # Convert to pandas DataFrame
        results_df = pd.DataFrame.from_records(results)
        return results_df
    except Exception as e:
        print(f"An error occurred: {e}")
        return pd.DataFrame()
    finally: 
        client.close()


In [12]:
temporal_query = {
    """
        SELECT
        date_trunc_ymd(transit_timestamp) AS date,
        CASE
            WHEN date_extract_dow(transit_timestamp) = 0 THEN 'Sunday'
            WHEN date_extract_dow(transit_timestamp) = 1 THEN 'Monday'
            WHEN date_extract_dow(transit_timestamp) = 2 THEN 'Tuesday'
            WHEN date_extract_dow(transit_timestamp) = 3 THEN 'Wednesday'
            WHEN date_extract_dow(transit_timestamp) = 4 THEN 'Thursday'
            WHEN date_extract_dow(transit_timestamp) = 5 THEN 'Friday'
            WHEN date_extract_dow(transit_timestamp) = 6 THEN 'Saturday'
        END AS day_of_week,
        date_extract_hh(transit_timestamp) AS hour_of_day,
        CASE
            -- Late night: midnight to 6:30 AM, every day
            WHEN (
                date_extract_hh(transit_timestamp) * 60
                + date_extract_mm(transit_timestamp)
            ) < 390
            THEN
                CASE
                    WHEN date_extract_dow(transit_timestamp) IN (0, 6)
                        THEN 'Weekend Late Night'
                    ELSE 'Weekday Late Night'
                END
            -- Weekend: 6:30 AM to midnight
            WHEN date_extract_dow(transit_timestamp) IN (0, 6)
                THEN 'Weekend Daytime/Evening'
            -- Weekday rush hours: 6:30-9:30 AM and 3:30-8 PM
            WHEN (
                date_extract_hh(transit_timestamp) * 60
                + date_extract_mm(transit_timestamp)
            ) >= 390
            AND (
                date_extract_hh(transit_timestamp) * 60
                + date_extract_mm(transit_timestamp)
            ) < 570
            THEN 'AM Rush Hours'
            WHEN (
                date_extract_hh(transit_timestamp) * 60
                + date_extract_mm(transit_timestamp)
            ) >= 930
            AND (
                date_extract_hh(transit_timestamp) * 60
                + date_extract_mm(transit_timestamp)
            ) < 1200
            THEN 'PM Rush Hours'
            -- Weekday midday: 9:30 AM to 3:30 PM
            WHEN (
                date_extract_hh(transit_timestamp) * 60
                + date_extract_mm(transit_timestamp)
            ) >= 570
            AND (
                date_extract_hh(transit_timestamp) * 60
                + date_extract_mm(transit_timestamp)
            ) < 930
            THEN 'Midday'
            -- Weekday evening: 8 PM to midnight
            ELSE 'Evening'
        END AS time_period,
        SUM(ridership) AS total_ridership
    WHERE transit_timestamp IS NOT NULL
    GROUP BY date, day_of_week, hour_of_day, time_period
    ORDER BY date, time_period
    LIMIT 1000000
    """
}

In [13]:
ridership_2025_2026 = query_open_data(ridership_data_sources["ridership_2025_2026"], temporal_query)
ridership_2025_2026["total_ridership"] = pd.to_numeric(ridership_2025_2026["total_ridership"])
print(ridership_2025_2026.shape)
ridership_2025_2026.to_parquet("ridership_2025_2026_temporal.parquet")

(15647, 5)


In [14]:
ridership_2020_2024 = query_open_data(ridership_data_sources["ridership_2020_2024"], temporal_query)
ridership_2020_2024["total_ridership"] = pd.to_numeric(ridership_2020_2024["total_ridership"])
print(ridership_2020_2024.shape)
ridership_2020_2024.to_parquet("ridership_2020_2024_temporal.parquet")

(43843, 5)


In [15]:
ridership_2017_2019 = query_open_data(ridership_data_sources["ridership_2017_2019"], temporal_query)
ridership_2017_2019["total_ridership"] = pd.to_numeric(ridership_2017_2019["total_ridership"])
print(ridership_2017_2019.shape)
ridership_2017_2019.to_parquet("ridership_2017_2019_temporal.parquet")

(25989, 5)


In [16]:
station_query = """
SELECT
    date_trunc_ymd(transit_timestamp) AS date,
    station_complex_id,
    station_complex,
    latitude,
    longitude,
    georeference,
    SUM(ridership) AS total_ridership
WHERE transit_timestamp IS NOT NULL
GROUP BY
    date,
    station_complex_id,
    station_complex,
    latitude,
    longitude,
    georeference
ORDER BY
    date,
    station_complex_id
LIMIT 10000000
"""

In [17]:
ridership_2025_2026_station = query_open_data(ridership_data_sources["ridership_2025_2026"], station_query)
ridership_2025_2026_station["total_ridership"] = pd.to_numeric(ridership_2025_2026_station["total_ridership"])
print(ridership_2025_2026_station)
ridership_2025_2026_station.to_parquet("ridership_2025_2026_station.parquet")

                           date station_complex_id  \
0       2025-01-01T00:00:00.000                  1   
1       2025-01-01T00:00:00.000                 10   
2       2025-01-01T00:00:00.000                100   
3       2025-01-01T00:00:00.000                101   
4       2025-01-01T00:00:00.000                103   
...                         ...                ...   
277565  2026-10-14T00:00:00.000                 97   
277566  2026-10-14T00:00:00.000                 98   
277567  2026-10-14T00:00:00.000                 99   
277568  2026-10-14T00:00:00.000              TRAM1   
277569  2026-10-14T00:00:00.000              TRAM2   

                   station_complex   latitude   longitude  \
0       Astoria-Ditmars Blvd (N,W)  40.775036   -73.91203   
1                    49 St (N,R,W)    40.7599   -73.98414   
2                   Hewes St (M,J)   40.70687   -73.95343   
3                 Marcy Av (M,J,Z)   40.70836  -73.957756   
4                     Bowery (J,Z)   40.72028 

In [20]:
ridership_2020_2024_station = query_open_data(ridership_data_sources["ridership_2020_2024"], station_query)
ridership_2020_2024_station["total_ridership"] = pd.to_numeric(ridership_2020_2024_station["total_ridership"])
print(ridership_2020_2024_station)
ridership_2020_2024_station.to_parquet("ridership_2020_2024_station.parquet")


                           date station_complex_id  \
0       2020-01-01T00:00:00.000                  1   
1       2020-01-01T00:00:00.000                 10   
2       2020-01-01T00:00:00.000                100   
3       2020-01-01T00:00:00.000                101   
4       2020-01-01T00:00:00.000                103   
...                         ...                ...   
875567  2024-12-31T00:00:00.000                 97   
875568  2024-12-31T00:00:00.000                 98   
875569  2024-12-31T00:00:00.000                 99   
875570  2024-12-31T00:00:00.000              TRAM1   
875571  2024-12-31T00:00:00.000              TRAM2   

                   station_complex   latitude   longitude  \
0       Astoria-Ditmars Blvd (N,W)  40.775036   -73.91203   
1                    49 St (N,R,W)    40.7599   -73.98414   
2                   Hewes St (M,J)   40.70687   -73.95343   
3                 Marcy Av (M,J,Z)   40.70836  -73.957756   
4                     Bowery (J,Z)   40.72028 

In [19]:
ridership_2017_2019_station = query_open_data(ridership_data_sources["ridership_2017_2019"], station_query)
ridership_2017_2019_station["total_ridership"] = pd.to_numeric(ridership_2017_2019_station["total_ridership"])
print(ridership_2017_2019_station) 
ridership_2017_2019_station.to_parquet("ridership_2017_2019_station.parquet")

                           date station_complex_id  \
0       2017-01-13T00:00:00.000                  1   
1       2017-01-13T00:00:00.000                 10   
2       2017-01-13T00:00:00.000                100   
3       2017-01-13T00:00:00.000                101   
4       2017-01-13T00:00:00.000                103   
...                         ...                ...   
509365  2019-12-31T00:00:00.000                 97   
509366  2019-12-31T00:00:00.000                 98   
509367  2019-12-31T00:00:00.000                 99   
509368  2019-12-31T00:00:00.000              TRAM1   
509369  2019-12-31T00:00:00.000              TRAM2   

                   station_complex   latitude   longitude  \
0       Astoria-Ditmars Blvd (N,W)  40.775036   -73.91203   
1                    49 St (N,R,W)    40.7599   -73.98414   
2                   Hewes St (M,J)   40.70687   -73.95343   
3                 Marcy Av (M,J,Z)   40.70836  -73.957756   
4                     Bowery (J,Z)   40.72028 

In [5]:
station_complex_source = '5f5g-n3cz'

station_complex_query = "SELECT *"

station_complexes = query_open_data(station_complex_source, station_complex_query)
station_complexes.to_parquet("station_complexes.parquet")

In [11]:
od_september_query = """SELECT * 
WHERE Month = 9
LIMIT 100000000"""

In [ ]:
od_2026_september = query_open_data(od_data_sources["od_2026"], od_month_day_sum_query)
od_2026_september["avg_daily_ridership"] = pd.to_numeric(od_2026_september["avg_daily_ridership"])
print(od_2026_september.shape)
od_2026_september.to_parquet("od_2026_september.parquet")


(10063706, 8)


In [13]:
od_2025_september = query_open_data(od_data_sources["od_2025"], od_month_day_sum_query)
od_2025_september["avg_daily_ridership"] = pd.to_numeric(od_2025_september["avg_daily_ridership"])
print(od_2025_september.shape)
od_2025_september.to_parquet("od_2025_september.parquet")


(13197899, 8)


In [14]:
od_2021_september = query_open_data(od_data_sources["od_2021"], od_month_day_sum_query)
od_2021_september["avg_daily_ridership"] = pd.to_numeric(od_2021_september["avg_daily_ridership"])
print(od_2021_september.shape)
od_2021_september.to_parquet("od_2021_september.parquet")


(12791506, 8)
